In [6]:
!pip install tensorflow

In [1]:
import argparse
import re
import shutil
from functools import partial
from pathlib import Path

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from lerobot.constants import HF_LEROBOT_HOME
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from oxe_utils.configs import OXE_DATASET_CONFIGS, ActionEncoding, StateEncoding
from oxe_utils.transforms import OXE_STANDARDIZATION_TRANSFORMS
from lerobot.datasets.utils import append_jsonlines
np.set_printoptions(precision=2)


import json
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import mediapy
from scipy.spatial.transform import Rotation as R
import cv2
import imageio

2026-01-05 13:40:46.344799: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-05 13:40:46.620035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767620446.642304  113538 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767620446.649062  113538 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767620446.666269  113538 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
# 指定本地缓存目录
ds = tfds.load(
    "droid",                # 数据集名称
    split="train",          # 数据集划分
    data_dir="/mnt/hwfile/tangyuhang"  # 本地缓存路径
)

I0000 00:00:1767620940.744184  113538 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79265 MB memory:  -> device: 0, name: NVIDIA A800-SXM4-80GB, pci bus id: 0000:c5:00.0, compute capability: 8.0


In [20]:
for example in ds.take(1):
    print("Keys:", example.keys())
    print("Episode metadata:")
    for k, v in example['episode_metadata'].items():
        print(f"  {k}: {v.numpy().decode('utf-8')}")
    print("Steps")
    for k, v in example['steps'].element_spec.items():
        print(f"  {k} : {v}")

Keys: dict_keys(['episode_metadata', 'steps'])
Episode metadata:
  file_path: gs://xembodiment_data/r2d2/r2d2-data-full/TRI/success/2024-02-08/Thu_Feb__8_16:38:30_2024/trajectory.h5
  recording_folderpath: gs://xembodiment_data/r2d2/r2d2-data-full/TRI/success/2024-02-08/Thu_Feb__8_16:38:30_2024/recordings/MP4
Steps
  action : TensorSpec(shape=(7,), dtype=tf.float64, name=None)
  action_dict : {'cartesian_position': TensorSpec(shape=(6,), dtype=tf.float64, name=None), 'cartesian_velocity': TensorSpec(shape=(6,), dtype=tf.float64, name=None), 'gripper_position': TensorSpec(shape=(1,), dtype=tf.float64, name=None), 'gripper_velocity': TensorSpec(shape=(1,), dtype=tf.float64, name=None), 'joint_position': TensorSpec(shape=(7,), dtype=tf.float64, name=None), 'joint_velocity': TensorSpec(shape=(7,), dtype=tf.float64, name=None)}
  discount : TensorSpec(shape=(), dtype=tf.float32, name=None)
  is_first : TensorSpec(shape=(), dtype=tf.bool, name=None)
  is_last : TensorSpec(shape=(), dtype=tf.

2026-01-05 14:01:29.007371: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [21]:
path_to_droid_repo = "/mnt/hwfile/tangyuhang/.cache/huggingface/hub/models--KarlP--droid/snapshots/bcb840c3b496533e0adf548a54b51f2f00057837" # TODO: Replace with the path to your DROID repository

# Load the extrinsics
cam2base_extrinsics_path = f"{path_to_droid_repo}/cam2base_extrinsics.json"
with open(cam2base_extrinsics_path, "r") as f:
    cam2base_extrinsics = json.load(f)

# Load the intrinsics
intrinsics_path = f"{path_to_droid_repo}/intrinsics.json"
with open(intrinsics_path, "r") as f:
    intrinsics = json.load(f)

# Load mapping from episode ID to path, then invert
episode_id_to_path_path = f"{path_to_droid_repo}/episode_id_to_path.json"
with open(episode_id_to_path_path, "r") as f:
    episode_id_to_path = json.load(f)
episode_path_to_id = {v: k for k, v in episode_id_to_path.items()}

# Load camera serials
camera_serials_path = f"{path_to_droid_repo}/camera_serials.json"
with open(camera_serials_path, "r") as f:
    camera_serials = json.load(f)

In [56]:
print(len(cam2base_extrinsics))
print(len(intrinsics))
print(len(episode_id_to_path))
print(len(camera_serials))


print("\ncam2base_extrinsics")
for k, v in list(cam2base_extrinsics.items())[:2]:
    print(f"\n  {k}: {v}")


print("\nintrinsics")
for k, v in list(intrinsics.items())[:2]:
    print(f"\n  {k}: {v}")

print("\nepisode id to path")
for k,v in list(episode_id_to_path.items())[:2]:
    print(f"\n  {k}: {v}")

print("\ncamera serials")
for k,v in list(camera_serials.items())[:2]:
    print(f"\n {k}: {v}")


36084
72468
74795
74795

cam2base_extrinsics

  AUTOLab+5d05c5aa+2023-07-07-09h-48m-37s: {'relative_path': 'AUTOLab/failure/2023-07-07/Fri_Jul__7_09:48:37_2023', '24400334': [0.2596757315060087, -0.36626259649963777, 0.24849304837972613, -1.742115402153725, -0.0012127426938948194, -0.7149867215760838], 'source': 'GT', 'metric_type': 'IoU', 'quality_metric': 0.7905524849515541}

  AUTOLab+5d05c5aa+2023-07-07-09h-50m-13s: {'relative_path': 'AUTOLab/failure/2023-07-07/Fri_Jul__7_09:50:13_2023', '24400334': [0.2596757315060087, -0.36626259649963777, 0.24849304837972613, -1.742115402153725, -0.0012127426938948194, -0.7149867215760838], 'source': 'GT', 'metric_type': 'IoU', 'quality_metric': 0.7429697221070095}

intrinsics

  AUTOLab+5d05c5aa+2023-07-07-10h-29m-59s: {'22008760': {'cameraMatrix': [524.4097290039062, 639.77783203125, 524.4097290039062, 370.2782897949219], 'distCoeffs': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'width': 1280, 'height': 720}, '18026681': {'ca

In [61]:
# Iterate through the dataset to find the first episode that has a cam2base extrinsics entry
eps = []
for ep in tqdm(ds):
    file_path = ep["episode_metadata"]["file_path"].numpy().decode("utf-8")
    recording_folderpath = ep["episode_metadata"]["recording_folderpath"].numpy().decode("utf-8")

    episode_path = file_path.split("r2d2-data-full/")[1].split("/trajectory")[0]
    if episode_path not in episode_path_to_id:
        continue
    episode_id = episode_path_to_id[episode_path]

    
    if episode_id in cam2base_extrinsics:
        eps.append(ep)
        print(f"\n episode_path: {episode_path}")
        print(f"\n episode_id: {episode_id}")    
        
    if len(eps) >= 1:
        break

  0%|          | 4/92233 [00:00<5:47:14,  4.43it/s]


 episode_path: WEIRD/success/2023-12-16/Sat_Dec_16_02:09:13_2023

 episode_id: WEIRD+f1c42455+2023-12-16-02h-09m-13s


  0%|          | 4/92233 [00:08<56:55:37,  2.22s/it]


In [62]:
# Iterate through the extrinsics to find key that is a digit
# This is the camera serial number, and the corresponding value is the extrinsics
for k, v in cam2base_extrinsics[episode_id].items():
    if k.isdigit():
        camera_serial = k
        extracted_extrinsics = v
        break

# Also lets us get the intrinsics
extracted_intrinsics = intrinsics[episode_id][camera_serial]

# Using the camera serial, find the corresponding camera name (which is used to determine
# which image stream in the episode to use)
camera_serials_to_name = {v: k for k, v in camera_serials[episode_id].items()}
calib_camera_name = camera_serials_to_name[camera_serial]

if calib_camera_name == "ext1_cam_serial":
    calib_image_name = "exterior_image_1_left"
elif calib_camera_name == "ext2_cam_serial":
    calib_image_name = "exterior_image_2_left"
else:
    raise ValueError(f"Unknown camera name: {calib_camera_name}")

print(f"Camera with calibration data: {calib_camera_name} --> {calib_image_name}")

Camera with calibration data: ext1_cam_serial --> exterior_image_1_left


In [63]:
# Convert the extrinsics to a homogeneous transformation matrix
pos = extracted_extrinsics[0:3] # translation
rot_mat = R.from_euler("xyz", extracted_extrinsics[3:6]).as_matrix() # rotation

# Make homogenous transformation matrix
cam_to_base_extrinsics_matrix = np.eye(4)
cam_to_base_extrinsics_matrix[:3, :3] = rot_mat
cam_to_base_extrinsics_matrix[:3, 3] = pos

print(cam_to_base_extrinsics_matrix)

[[ 0.55 -0.17  0.82 -0.18]
 [-0.83 -0.05  0.55 -0.51]
 [-0.05 -0.98 -0.17  0.51]
 [ 0.    0.    0.    1.  ]]


In [64]:
# Convert the intrinsics to a matrix
fx, cx, fy, cy = extracted_intrinsics["cameraMatrix"]
intrinsics_matrix = np.array([
        [fx, 0, cx],
        [0, fy, cy],
        [0, 0, 1]
])
print(intrinsics_matrix)

[[523.84   0.   636.31]
 [  0.   523.84 368.03]
 [  0.     0.     1.  ]]


In [65]:
# Save all observations for the calibrated camera and corresponding gripper positions
images = []
cartesian_poses = []
for step in ep["steps"]:
    image = step["observation"][calib_image_name].numpy()
    images.append(image)
    cartesian_pose = step["observation"]["cartesian_position"].numpy()
    cartesian_poses.append(cartesian_pose)

# length images x 6
cartesian_poses = np.array(cartesian_poses)
# Remove the rotation and make homogeneous: --> length images x 3 --> length images x 4
cartesian_homogeneous_positions = cartesian_poses[:, :3]
cartesian_homogeneous_positions = np.hstack(
    (cartesian_homogeneous_positions, np.ones((cartesian_homogeneous_positions.shape[0], 1)))
)

# Transpose to support matrix multiplication: --> 4 x length images
gripper_position_base = cartesian_homogeneous_positions.T

In [77]:
print(len(images) == len(cartesian_poses))

print("\ncartesian homogeneous popsitions\n")
print(cartesian_homogeneous_positions)

print("\ngripper position base\n")
print(gripper_position_base)

True

cartesian homogeneous popsitions

[[ 0.45 -0.1   0.45  1.  ]
 [ 0.45 -0.1   0.45  1.  ]
 [ 0.45 -0.1   0.45  1.  ]
 ...
 [ 0.57 -0.02  0.49  1.  ]
 [ 0.56 -0.03  0.47  1.  ]
 [ 0.56 -0.03  0.47  1.  ]]

gripper position base

[[ 0.45  0.45  0.45 ...  0.57  0.56  0.56]
 [-0.1  -0.1  -0.1  ... -0.02 -0.03 -0.03]
 [ 0.45  0.45  0.45 ...  0.49  0.47  0.47]
 [ 1.    1.    1.   ...  1.    1.    1.  ]]


In [79]:
# Transform gripper position to camera frame, then remove homogeneous component
base_to_cam_extrinsics_matrix = np.linalg.inv(cam_to_base_extrinsics_matrix)
robot_gripper_position_cam = base_to_cam_extrinsics_matrix @ gripper_position_base
robot_gripper_position_cam = robot_gripper_position_cam[:3] # Now 3 x length images

In [80]:
# Finally, use intrinsics to project the gripper position in camera frame into pixel space
pixel_positions = intrinsics_matrix @ robot_gripper_position_cam[:3]
pixel_positions = pixel_positions[:2] / pixel_positions[2]

In [81]:
# Visualize!
vis_images = []
temp_img_path = f"{path_to_droid_repo}/TEMP.png"

for i, image in enumerate(tqdm(images)):
    if i % 10 != 0:
        continue
    
    fig, axs = plt.subplots(1)
    x, y = pixel_positions[0, i] / 1280 * 320, pixel_positions[1, i] / 720 * 180 # Scale to match image dimensions

    # clip coords
    x = np.clip(x, 0, 320)
    y = np.clip(y, 0, 180)

    axs.imshow(image)
    axs.scatter(x, y, c='red', s=20)
    axs.set_xlim(0, 320)
    axs.set_ylim(180, 0)  # Invert y-axis to match image

    # turn off axes
    axs.axis('off')

    # save the figure, then reopen it as PIL image
    plt.savefig(temp_img_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    vis_image = Image.open(temp_img_path).convert("RGB")
    vis_images.append(np.array(vis_image))

100%|██████████| 283/283 [00:05<00:00, 48.87it/s]


In [82]:
# Visualize the video
mediapy.show_video(
    vis_images,
    fps=8
)

This browser does not support the video tag.
